# RAG Pipeline: Retrieval-Augmented Generation

Build RAG system combining hybrid retrieval with LLM for question answering. Auto-generates supporting modules for document chunking, prompts, and pipeline orchestration.

## Output Files
- **src/chunking.py** - DocumentChunker class (auto-generated)
- **src/prompts.py** - RAGPrompts class with templates (auto-generated)
- **src/rag_pipeline.py** - RAGPipeline class with hybrid retrieval (auto-generated)

## Workflow
1. Setup project paths
2. Create directories
3. Auto-generate src/chunking.py for document chunking
4. Auto-generate src/prompts.py with prompt templates
5. Auto-generate src/rag_pipeline.py with RAG orchestration
6. Verify files generated
7. Evaluate RAG pipeline on 10 queries (Easy / Medium / Complex)

## 1. Setup Project Paths

Detects if running from notebooks/ folder and sets project_root correctly. Must run BEFORE any imports from src.

**Output:** Project root path confirmation

In [1]:
from pathlib import Path
import sys

notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
    print("Detected: Running from notebooks/ folder")
else:
    project_root = notebook_dir
    print("Detected: Running from project root")

sys.path.insert(0, str(project_root))

print(f"\nProject root: {project_root}")
print(f"Sys path updated")
print("\nReady for file generation")

Detected: Running from notebooks/ folder

Project root: /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki
Sys path updated

Ready for file generation


## 2. Create Directories

Creates src/ and data/processed directories if they don't exist.

**Output:** Directory confirmation

In [2]:
src_dir = project_root / "src"
data_dir = project_root / "data" / "processed"

src_dir.mkdir(exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

print(f"Directories ready:")
print(f"  src/: {src_dir.exists()}")
print(f"  data/processed/: {data_dir.exists()}")

Directories ready:
  src/: True
  data/processed/: True


## 3. Auto-Generate src/chunking.py

Creates DocumentChunker class for splitting long documents into overlapping chunks.

**Output:** src/chunking.py file created with DocumentChunker class

In [3]:
chunking_code = '''from typing import List


class DocumentChunker:
    """Chunk documents into manageable pieces."""
    
    def __init__(self, chunk_size: int = 500, overlap: int = 50):
        """Initialize chunker with size and overlap parameters."""
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def chunk_document(self, text: str) -> List[str]:
        """Split document into chunks with overlap."""
        if len(text) <= self.chunk_size:
            return [text]
        
        chunks = []
        start = 0
        
        while start < len(text):
            end = min(start + self.chunk_size, len(text))
            chunks.append(text[start:end])
            start = end - self.overlap
        
        return chunks
'''

chunking_path = project_root / "src" / "chunking.py"
with open(chunking_path, 'w') as f:
    f.write(chunking_code)

print(f"Created: src/chunking.py")
print(f"  Class: DocumentChunker")

Created: src/chunking.py
  Class: DocumentChunker


## 4. Auto-Generate src/prompts.py

Creates RAGPrompts class with configurable prompt templates for different answer styles.

**Output:** src/prompts.py file created with RAGPrompts class

In [4]:
prompts_code = '''class RAGPrompts:
    """RAG prompt templates for different answer styles."""
    
    BALANCED = """Based on the following book reviews and information, answer the question.

Context:
{context}

Question: {question}

Answer:"""
    
    STRICT = """Using ONLY the provided book reviews and information below, answer the question.
If the information is insufficient to answer, say so.

Context:
{context}

Question: {question}

Answer:"""
    
    @staticmethod
    def get_template(version: str = "balanced") -> str:
        """Get prompt template by version name."""
        templates = {
            "balanced": RAGPrompts.BALANCED,
            "strict": RAGPrompts.STRICT
        }
        return templates.get(version.lower(), RAGPrompts.BALANCED)
'''

prompts_path = project_root / "src" / "prompts.py"
with open(prompts_path, 'w') as f:
    f.write(prompts_code)

print(f"Created: src/prompts.py")
print(f"  Class: RAGPrompts")
print(f"  Templates: balanced, strict")

Created: src/prompts.py
  Class: RAGPrompts
  Templates: balanced, strict


## 5. Auto-Generate src/rag_pipeline.py

Creates RAGPipeline class orchestrating complete RAG workflow.

**Output:** src/rag_pipeline.py file created with RAGPipeline class

In [5]:
rag_pipeline_code = '''from typing import List, Dict, Tuple
from src.chunking import DocumentChunker
from src.prompts import RAGPrompts
from src.hybrid import HybridRetriever


class RAGPipeline:
    """RAG pipeline using HybridRetriever for retrieval and an LLM for generation."""

    def __init__(self, bm25_retriever, semantic_retriever, llm, prompt_version="balanced"):
        """Initialize RAG pipeline with retrieval components and LLM."""
        self.hybrid = HybridRetriever(bm25_retriever, semantic_retriever)
        self.llm = llm
        self.corpus = None
        self.chunker = DocumentChunker(chunk_size=500, overlap=50)
        self.prompt_template = RAGPrompts.get_template(prompt_version)

    def retrieve_hybrid(self, query: str, top_k: int = 5) -> Tuple[List[int], List[str]]:
        """Retrieve top-k documents using the hybrid retriever."""
        results = self.hybrid.search(query, top_k)
        doc_ids = [doc_id for doc_id, _ in results]
        documents = [self.corpus[doc_id] for doc_id in doc_ids]
        return doc_ids, documents

    def build_context(self, documents: List[str], max_tokens: int = 2000) -> str:
        """Build context from documents with token limit."""
        context = ""
        token_count = 0

        for i, doc in enumerate(documents, 1):
            review_text = "Review " + str(i) + ": " + doc
            token_count += len(review_text.split())

            if token_count > max_tokens:
                break

            context += review_text + " "

        return context

    def generate(self, query: str, context: str) -> str:
        """Generate answer using LLM."""
        prompt = self.prompt_template.format(context=context, question=query)

        try:
            return self.llm.invoke(prompt)
        except Exception as e:
            return "Error: " + str(e)

    def invoke(self, query: str, top_k: int = 5) -> Dict:
        """Complete RAG pipeline: retrieve -> context -> generate."""
        doc_ids, documents = self.retrieve_hybrid(query, top_k)
        context = self.build_context(documents)
        answer = self.generate(query, context)

        return {
            "query": query,
            "documents_retrieved": len(documents),
            "context_length": len(context),
            "answer": answer,
            "retrieved_doc_ids": doc_ids
        }
'''

rag_path = project_root / "src" / "rag_pipeline.py"
with open(rag_path, 'w') as f:
    f.write(rag_pipeline_code)

print(f"Created: src/rag_pipeline.py")
print(f"  Class: RAGPipeline")
print(f"  Methods: retrieve_hybrid, build_context, generate, invoke")


Created: src/rag_pipeline.py
  Class: RAGPipeline
  Methods: retrieve_hybrid, build_context, generate, invoke


## 6. Verify All Files Generated

Checks that all 3 modules were created successfully WITHOUT importing them (lightweight verification).

**Output:** File list - NO kernel crash (memory optimized for MacBook)

In [6]:
import os

files_to_check = [
    (project_root / "src" / "chunking.py", "DocumentChunker"),
    (project_root / "src" / "prompts.py", "RAGPrompts"),
    (project_root / "src" / "rag_pipeline.py", "RAGPipeline")
]

print("Generated Files Summary")
print("=" * 60)

all_exist = True
for file_path, class_name in files_to_check:
    if file_path.exists():
        size = os.path.getsize(file_path)
        print(f"\nOK: {file_path.name}")
        print(f"  Class: {class_name}")
        print(f"  Size: {size} bytes")
    else:
        print(f"\nMISSING: {file_path.name}")
        all_exist = False

print()
print("=" * 60)
if all_exist:
    print("All RAG files generated successfully")
    print("\nReady to use with app.py")
else:
    print("Some files failed to generate")

Generated Files Summary

OK: chunking.py
  Class: DocumentChunker
  Size: 714 bytes

OK: prompts.py
  Class: RAGPrompts
  Size: 741 bytes

OK: rag_pipeline.py
  Class: RAGPipeline
  Size: 2336 bytes

All RAG files generated successfully

Ready to use with app.py


## 7. RAG Pipeline Evaluation

Runs the full Hybrid RAG pipeline on the same 10 queries used in notebook 05, so results can be compared across BM25, Semantic, and RAG. Uses Groq LLM if `GROQ_API_KEY` is set in `.env`, otherwise falls back to SimpleLLM.

Difficulty levels: **E** = Easy (keyword), **M** = Medium (semantic), **C** = Complex.

In [7]:
import pickle
import os
from dotenv import load_dotenv

load_dotenv()

from src.bm25 import BM25Retriever
from src.semantic_retriever import SemanticRetriever
from src.rag_pipeline import RAGPipeline

data_dir = project_root / 'data' / 'processed'

with open(data_dir / 'corpus.pkl', 'rb') as f:
    corpus = pickle.load(f)

bm25 = BM25Retriever()
bm25.load(str(data_dir / 'bm25_index.pkl'))
bm25.corpus = corpus

semantic = SemanticRetriever()
semantic.load(str(data_dir / 'semantic_index' / 'faiss_index'))
semantic.corpus = corpus

# Use Groq if API key is available, otherwise use SimpleLLM
api_key = os.getenv('GROQ_API_KEY')
if api_key:
    try:
        from groq import Groq
        class GroqLLM:
            MODELS = ['llama-3.2-90b-vision-preview', 'llama-3.1-70b-versatile', 'llama-3.1-8b-instant', 'gemma2-9b-it']
            def __init__(self): self.client = Groq(api_key=api_key)
            def invoke(self, prompt):
                for model in self.MODELS:
                    try:
                        r = self.client.chat.completions.create(
                            messages=[{'role': 'user', 'content': prompt}],
                            model=model, max_tokens=300, temperature=0.3)
                        return r.choices[0].message.content
                    except Exception as e:
                        if 'decommissioned' in str(e) or 'not supported' in str(e): continue
                        return f'Error: {e}'
                return 'All models failed.'
        llm = GroqLLM()
        print('LLM: Groq (llama-3.2-90b-vision-preview with fallback)')
    except ImportError:
        class SimpleLLM:
            def invoke(self, prompt): return '[SimpleLLM placeholder]'
        llm = SimpleLLM()
        print('LLM: SimpleLLM (groq package not installed)')
else:
    class SimpleLLM:
        def invoke(self, prompt): return '[SimpleLLM placeholder — set GROQ_API_KEY for real answers]'
    llm = SimpleLLM()
    print('LLM: SimpleLLM (no GROQ_API_KEY found)')

rag = RAGPipeline(bm25, semantic, llm)
rag.corpus = corpus
print(f'Corpus: {len(corpus):,} documents')
print('RAG pipeline ready')


BM25 index loaded from /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki/data/processed/bm25_index.pkl


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index loaded from /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki/data/processed/semantic_index/faiss_index
LLM: SimpleLLM (no GROQ_API_KEY found)
Corpus: 91,850 documents
RAG pipeline ready


In [8]:
import pandas as pd

df = pd.read_parquet(data_dir / 'books_sample.parquet')

test_queries = [
    # Easy - keyword-based
    'mystery novel',
    'cookbook recipes',
    'science fiction space',
    'python programming',
    # Medium - semantic
    'book to help with anxiety',
    'story about finding yourself',
    'guide for first time parents',
    # Complex
    'best book to learn machine learning with no math background',
    'historical fiction set in world war 2 from a female perspective',
    'self help book for overcoming procrastination and building better habits',
]

difficulty = ['E', 'E', 'E', 'E', 'M', 'M', 'M', 'C', 'C', 'C']

print('=' * 70)
print('RAG Pipeline Evaluation')
print('=' * 70)

for query, diff in zip(test_queries, difficulty):
    result = rag.invoke(query, top_k=5)
    print(f'\nQuery [{diff}]: "{query}"')
    print(f'Retrieved {result["documents_retrieved"]} docs | Context: {result["context_length"]} chars')
    print('Retrieved books:')
    for rank, doc_id in enumerate(result['retrieved_doc_ids'], 1):
        title = df.iloc[doc_id]['product_title']
        rating = df.iloc[doc_id]['rating']
        print(f'  {rank}. {title} ({rating}/5)')
    print('Answer:')
    print(f'  {result["answer"]}')
    print('-' * 70)


RAG Pipeline Evaluation

Query [E]: "mystery novel"
Retrieved 5 docs | Context: 1546 chars
Retrieved books:
  1. The Woman in the Window: A Novel (5.0/5)
  2. And Then There Were None (5.0/5)
  3. One Little Wish: a romantic suspense novel (The Little Things Mystery Series) (5.0/5)
  4. Mystery: An Alex Delaware Novel (3.0/5)
  5. The Last Lie Told (Finley O’Sullivan Book 1) (5.0/5)
Answer:
  [SimpleLLM placeholder — set GROQ_API_KEY for real answers]
----------------------------------------------------------------------

Query [E]: "cookbook recipes"
Retrieved 5 docs | Context: 1627 chars
Retrieved books:
  1. Japanese Cookbook for Beginners: 2 Books in 1, Sushi Cookbook + Ramen Cookbook, Quick and Easy Japanese Recipes to Make a Perfect Dinner at Home (Maggie Barton's Recipe Books) (5.0/5)
  2. The Complete Cooking for Two Cookbook: 650 Recipes for Everything You'll Ever Want to Make (The Complete ATK Cookbook Series) (5.0/5)
  3. Tacos: Recipes and Provocations: A Cookbook (5.0/5)
 